In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

**<font size="6" color="red">ch2. Ollama_LLM활용의 기본 개념(LangChain)</font>**

# 1. LLM을 활용하여 답변 생성
## 1) Ollama 이용한 로컬 LLM 이용
- 성능은 GPT(open ai API), Claude같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용

### ollama.com 설치 -> 모델 pull
- cmd창에서 ollama pull deepseek-r1:1.5b

In [1]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='deepseek-r1:1.5b')
result = llm.invoke('What is the capital of France?')
result # AIMessage
# content : 실제 답변
# response_metadata : 모델 실행에 대한 상세 정보(전체소요시간, 모델로딩시간, 처리토큰수)

AIMessage(content='\n\nThe capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-09-11T00:59:42.8680392Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12147599200, 'load_duration': 1624569900, 'prompt_eval_count': 10, 'prompt_eval_duration': 99440000, 'eval_count': 355, 'eval_duration': 10416874000, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a08dfa-101c-7272-90b5-5bcf162da2ac-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 355, 'total_tokens': 365})

In [2]:
print(result.content)



The capital of France is Paris.


### 모델 pull
- ollama pull llama3.2:1b
- ollama 모델은 공식적으로 한글지원 안 됨(llama3.1:405b 한글지원 가능 -> llama3.2:3b한글 지원이 일부)
- exaone 모델은 공식적으로 한글지원 : ollama pull exaone3.5:2.4b

- 모델 저장 경로 : C:\Users\내컴퓨터이름\.ollama

In [3]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b',
                 temperature=0.2,
                 top_k=40, 
                 top_p=0.9, 
                 # num_ctx=4096 # 컨텍스트 윈도우(입력토큰과 출력토큰)
                )
result = llm.invoke('What is the capital of Korea?')
result

AIMessage(content='The capital of South Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T00:59:51.0501169Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2220698000, 'load_duration': 1607913500, 'prompt_eval_count': 32, 'prompt_eval_duration': 307981000, 'eval_count': 9, 'eval_duration': 300319000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08dfa-56dc-7093-8216-030d9340ddac-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 9, 'total_tokens': 41})

In [4]:
result.content

'The capital of South Korea is Seoul.'

In [5]:
llm = ChatOllama(model='exaone3.5:2.4b')
result = llm.invoke('한국 수도는 어디예요?')
result.content

'한국의 수도는 **서울**입니다. 서울은 정치, 경제, 문화의 중심지로 국가의 행정과 중요한 기관들이 위치해 있습니다.'

## 2) openai 모델 활용
- pip install langchain-openai

In [6]:
# 환경변수('OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable')
from dotenv import load_dotenv
import os
load_dotenv()
# print(os.getenv('OPENAI_API_KEY'))
# print(os.environ['OPENAI_API_KEY'])

True

In [8]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano", 
                 # api_key=os.getenv('OPENAI_API_KEY')
                )
result = llm.invoke('What is the capital of Korea?')
# result = llm.invoke('한국의 수도가 어디에요?')
result.content

'The capital of South Korea is Seoul.'

In [12]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model='claude-haiku-4-5-20251001')
# ANTHROPIC_API_KEY environment variable
# llm.invoke('What is the capital of Korea?')

# 2. 랭체인 스타일로 프롬프트 작성하기
- 프롬프트 : llm호출시 쓰는 질문

## 1) 기본 프롬프트 템플릿 사용
- PromptTemplate 을 사용하여 변수가 포함된 템플릿 작성하면 PromptValue를 만들 수 있다

In [18]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
# llm.invoke(0)
llm.invoke("What is the capital of Korea")
# 프롬프트 가능 타입 : str, PromptValue, list of BaseMessages

AIMessage(content='The capital of Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T02:16:18.8138088Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2243330800, 'load_duration': 1622502600, 'prompt_eval_count': 31, 'prompt_eval_duration': 347440000, 'eval_count': 8, 'eval_duration': 266228000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e40-57b8-70c1-afa5-5c0cf2176f4e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 8, 'total_tokens': 39})

In [18]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(
                        template="What is the capital of {country}?", # {}안에 값을 새로운 값으로 대체
                        input_variables = ['country']
            )
prompt = prompt_template.invoke({"country":"Korea"})
print(1,prompt)
prompt = prompt_template.invoke("Korea")
print(2,prompt)
llm.invoke(prompt)

1 text='What is the capital of Korea?'
2 text='What is the capital of Korea?'


AIMessage(content="The capital of South Korea is Seoul. However, the official administrative capital is Incheon, which has served as the country's primary hub for government and business activities since 1987.", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:59:01.2983168Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1590585100, 'load_duration': 3801400, 'prompt_eval_count': 32, 'prompt_eval_duration': 201559000, 'eval_count': 38, 'eval_duration': 1379981000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e9e-627a-7a32-9dd5-33ad4ca02f31-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 38, 'total_tokens': 70})

In [ ]:
country = input('수도를 알고 싶은 나라는(영어)?')
llm.invoke(prompt_template.invoke(country))

In [2]:
def answer(country):
    '나라명을 입력받아 수도를 llm에게 수도명을 받아 return'
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate
    llm = ChatOllama(model='llama3.2:1b')
    prompt_template = PromptTemplate(
                        template='What is the capital of {country}?',
                        input_variables = ['country']
                    )
    result = llm.invoke(prompt_template.invoke(country))
    return result.content

In [3]:
country = input("수도를 알고 싶은 나라는(영어)>")
answer(country)

수도를 알고 싶은 나라는(영어)>turkey


'The capital of Turkey is Ankara.'

## 2) 메세지 기반 프롬프트 작성
- list of BaseMessages
- BaseMessage 상속받은 클래스 : AIMessage, HumanMessage, SystemMessage, ToolMessage
- [BaseMessage객체, BaseMessage객체, BaseMessage객체, ...]

In [6]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
message_list = [
    SystemMessage(content="You are a helpful assistant!"), # llm 페르소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Rome.'),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content='The capital of Italy is Paris.'),
    HumanMessage(content="What is the capital of Korea?") # llm에게 질문하고 싶은 진짜 내용
]
llm.invoke(message_list)

AIMessage(content='The capital of South Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:35:28.0172963Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2771673500, 'load_duration': 1604084700, 'prompt_eval_count': 86, 'prompt_eval_duration': 851813000, 'eval_count': 9, 'eval_duration': 310731000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e88-cd3c-7b60-a7ea-1a00e233f979-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 9, 'total_tokens': 95})

## 3) ChatPromptTemplate 사용(추천; 확장성 용이)

In [33]:
from langchain_core.prompts import ChatPromptTemplate
ChatPromptTemplate = ChatPromptTemplate([
    SystemMessage(content="You are a helpful assistant!"), # llm 페르소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Rome.'),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content='The capital of Italy is Paris.'),
    HumanMessage(content="What is the capital of Korea?") # llm에게 질문하고 싶은 진짜 내용
])
prompt = ChatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 :', prompt)

프롬프트 : messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={})]


In [13]:
from langchain_core.prompts import ChatPromptTemplate
ChatPromptTemplate = ChatPromptTemplate([
    ("system", "You are a helpful assistant!"), 
    ("human", "What is the capital of Italy?"),
    ("ai", "The capital of Italy is Rome."),
    ("human", "What is the capital of France?"),
    ("ai", "The capital of France is Paris."),
    ("human", "What is the capital of {country}?")
    # SystemMessage(content="You are a helpful assistant!"), # llm 페르소나
    # HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    # AIMessage(content='The capital of Italy is Rome.'),
    # HumanMessage(content="What is the capital of France?"),
    # AIMessage(content='The capital of Italy is Paris.'),
    # HumanMessage(content="What is the capital of Korea?") # llm에게 질문하고 싶은 진짜 내용
])
prompt = ChatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 :', prompt)

프롬프트 : messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={})]


In [11]:
llm.invoke(prompt)

AIMessage(content='The capital of South Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T03:50:59.4340239Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2721237100, 'load_duration': 1617456300, 'prompt_eval_count': 86, 'prompt_eval_duration': 806112000, 'eval_count': 9, 'eval_duration': 292949000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08e97-03c6-7fd1-93f4-4dffbc5b4837-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 9, 'total_tokens': 95})

In [16]:
# llm.invoke(chatPromptTemplate.invoke({'country':'Korea'}))
llm.invoke(chatPromptTemplate.invoke({'Korea'}))

NameError: name 'chatPromptTemplate' is not defined

# 3. 답변 형식 컨트롤하기
- invoke 실행 결과 AIMessage() -> String, json 변환해주는 OutputParser 이용

## 1) 문자열 출력 파서 이용
- StrOutputParser를 이용하여 LLM출력(AIMessage)를 단순 문자열로 변환

In [25]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables = ['country']
)
# 프롬프트 템플릿에 값 주입
prompt = prompt_template.invoke({'country':'Korea'})
print('프롬프트 :', prompt)
# llm에 질문
aimessage = llm.invoke(prompt)
# aimessage중 답변만 문자로 받기
output_parser = StrOutputParser()
result = output_parser.invoke(aimessage)
print('파서 결과 :', result)

프롬프트 : text='What is the capital of Korea? Return the name of the city only.'
파서 결과 : Seoul


In [27]:
output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))
output_parser.invoke(llm.invoke(prompt_template.invoke('Korea')))

'Seoul'

In [35]:
chatPromptTemplate = ChatPromptTemplate([
    ("system", "You are a helpful assistant!"), 
    ("human", "What is the capital of Italy?"),
    ("ai", "The capital of Italy is Rome."),
    ("human", "What is the capital of France?"),
    ("ai", "The capital of France is Paris."),
    ("human", "What is the capital of {country}? Return the name of the city only.")
])
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(chatPromptTemplate.invoke({'country':'Korea'})))

TypeError: 'ChatPromptTemplate' object is not callable

## 2) Json 출력파서 이용